In [ ]:

import pandas as pd
import numpy as np
import re

# ==========================================
# SPENDDNA - PERSONAL FINANCE ANALYSIS
# ==========================================

print("=" * 55)
print("           SPENDDNA TRANSACTION ANALYSIS")
print("=" * 55)

# Load dataset
df = pd.read_csv("transactions.csv")

# ------------------------------------------
# 1. DATA CLEANING
# ------------------------------------------

df.columns = df.columns.str.strip().str.lower()

# Expected columns: date, amount, type, description

df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)

df["amount"] = (
    df["amount"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₹", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df["type"] = df["type"].astype(str).str.upper().str.strip()

df["description"] = df["description"].fillna("").astype(str)

df = df.dropna(subset=["date", "amount"])
df = df.drop_duplicates()

df = df[df["amount"] > 0].copy()

# ------------------------------------------
# 2. VENDOR EXTRACTION
# ------------------------------------------

def extract_vendor(description):

    text = description.upper()

    vendors = {
        "SWIGGY": ["SWIGGY", "BUNDL"],
        "ZOMATO": ["ZOMATO"],
        "AMAZON": ["AMAZON"],
        "BLINKIT": ["BLINKIT", "GROFERS"],
        "ZEPTO": ["ZEPTO"],
        "UBER": ["UBER"],
        "OLA": ["OLA"],
        "NETFLIX": ["NETFLIX"],
        "SPOTIFY": ["SPOTIFY"],
        "BIGBASKET": ["BIGBASKET"],
        "Zerodha": ["ZERODHA"],
        "GROWW": ["GROWW"],
        "JIO": ["JIO"],
        "AIRTEL": ["AIRTEL"],
        "DMART": ["DMART"]
    }

    for vendor, keywords in vendors.items():
        if any(word in text for word in keywords):
            return vendor

    return "OTHER"


df["vendor"] = df["description"].apply(extract_vendor)

# ------------------------------------------
# 3. EXPENSE CATEGORY
# ------------------------------------------

def categorize(vendor):

    categories = {
        "SWIGGY": "Food Delivery",
        "ZOMATO": "Food Delivery",
        "BLINKIT": "Quick Commerce",
        "ZEPTO": "Quick Commerce",
        "AMAZON": "E-commerce",
        "UBER": "Transport",
        "OLA": "Transport",
        "NETFLIX": "Entertainment",
        "SPOTIFY": "Entertainment",
        "BIGBASKET": "Groceries",
        "DMART": "Groceries",
        "Zerodha": "Investments",
        "GROWW": "Investments",
        "JIO": "Utilities",
        "AIRTEL": "Utilities"
    }

    return categories.get(vendor, "Others")


df["category"] = df["vendor"].apply(categorize)

# ------------------------------------------
# 4. CREDIT AND DEBIT ANALYSIS
# ------------------------------------------

debits = df[df["type"].str.contains("DEBIT", na=False)]
credits = df[df["type"].str.contains("CREDIT", na=False)]

total_debit = debits["amount"].sum()
total_credit = credits["amount"].sum()

net_savings = total_credit - total_debit

savings_rate = (
    net_savings / total_credit * 100
    if total_credit > 0 else 0
)

print("\nSPENDING OVERVIEW")
print("-" * 40)

print("Total Transactions:", len(df))
print("Total Credits: ₹", round(total_credit, 2))
print("Total Debits: ₹", round(total_debit, 2))
print("Net Savings: ₹", round(net_savings, 2))
print("Savings Rate:", round(savings_rate, 2), "%")

# ------------------------------------------
# 5. CATEGORY-WISE SPENDING
# ------------------------------------------

print("\nCATEGORY-WISE SPENDING")
print("-" * 40)

category_spending = debits.groupby("category")["amount"].sum()

print(category_spending.sort_values(ascending=False))

# ------------------------------------------
# 6. VENDOR-WISE SPENDING
# ------------------------------------------

print("\nTOP VENDORS")
print("-" * 40)

vendor_spending = debits.groupby("vendor")["amount"].sum()

print(vendor_spending.sort_values(ascending=False).head(10))

# ------------------------------------------
# 7. MONTHLY TREND ANALYSIS
# ------------------------------------------

debits = debits.copy()

debits["month"] = debits["date"].dt.to_period("M")

monthly_spending = debits.groupby("month")["amount"].sum()

print("\nMONTHLY SPENDING")
print("-" * 40)

print(monthly_spending)

# ------------------------------------------
# 8. TIME OF DAY ANALYSIS
# ------------------------------------------

df["hour"] = df["date"].dt.hour

def time_period(hour):

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


df["time_period"] = df["hour"].apply(time_period)

print("\nTIME OF DAY ANALYSIS")
print("-" * 40)

print(df.groupby("time_period").size())

# ------------------------------------------
# 9. ANOMALY DETECTION USING Z-SCORE
# ------------------------------------------

def detect_anomalies(group):

    mean = group["amount"].mean()
    std = group["amount"].std()

    if pd.isna(std) or std == 0:
        return pd.Series(False, index=group.index)

    z_score = (group["amount"] - mean) / std

    return z_score.abs() > 2


debits["anomaly"] = (
    debits.groupby("category", group_keys=False)
    .apply(detect_anomalies)
)

anomalies = debits[debits["anomaly"]]

print("\nUNUSUAL TRANSACTIONS")
print("-" * 40)

print(anomalies[["date", "vendor", "category", "amount"]])

# ------------------------------------------
# 10. FINANCIAL PERSONALITY
# ------------------------------------------

print("\nYOUR SPENDING PERSONALITY")
print("-" * 40)

personalities = []

food_spending = category_spending.get("Food Delivery", 0)
shopping_spending = category_spending.get("E-commerce", 0)
investment_spending = category_spending.get("Investments", 0)

if total_debit > 0:

    if food_spending / total_debit > 0.20:
        personalities.append("THE FOODIE")

    if shopping_spending / total_debit > 0.25:
        personalities.append("THE SHOPAHOLIC")

    if investment_spending / total_debit > 0.20:
        personalities.append("THE INVESTOR")

    if savings_rate > 20:
        personalities.append("THE DISCIPLINED SAVER")

    if not personalities:
        personalities.append("THE BALANCED SPENDER")

for personality in personalities:
    print("★", personality)

# ------------------------------------------
# 11. FINAL REPORT
# ------------------------------------------

print("\n" + "=" * 55)
print("              FINAL SPENDDNA REPORT")
print("=" * 55)

print("Transactions:", len(df))
print("Credits: ₹", round(total_credit, 2))
print("Expenses: ₹", round(total_debit, 2))
print("Savings: ₹", round(net_savings, 2))

print("\nTop Spending Categories:")
print(category_spending.sort_values(ascending=False).head(5))

print("\nFinancial Personality:")
print(", ".join(personalities))

# ------------------------------------------
# 12. EXPORT RESULTS
# ------------------------------------------

df.to_csv("SpendDNA_Final.csv", index=False)

anomalies.to_csv("SpendDNA_Anomalies.csv", index=False)

print("\nFiles exported successfully!")
print("1. SpendDNA_Final.csv")
print("2. SpendDNA_Anomalies.csv")

print("\nSPENDDNA ANALYSIS COMPLETED!")
